# PS S6E4 - GBDT CATNUM top4 minimum

目的:
- high-IV OVR scan の top4 categorical columns を起点に、
  単カテゴリ CATNUM を最小構成で追加する
- まずは CatBoost で効くかを確認する
- pair CATNUM / TE / group interaction はまだやらない

対象カテゴリ:
- Crop_Growth_Stage
- Water_Source
- Crop_Type
- Irrigation_Type

## 1. Imports / config

In [1]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_SPLITS = 5

    EXP_NAME = "catboost_catnum_top4_min"
    SAVE_DIR = Path("/kaggle/working/exp_20260406_013_gbdt_catnum_top4_min")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    HIGH_IV_TOP4 = [
        "Crop_Growth_Stage",
        "Water_Source",
        "Crop_Type",
        "Irrigation_Type",
    ]

    NUM_COLS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    BASE_CAT_COLS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Region",
        "Mulching_Used",
    ]

    CATNUM_STATS = ["mean", "std", "median", "min", "max"]
    FILLNA_VALUE = -9999.0

    # CFG から削除
    # CLASS_ORDER = ["Low", "Medium", "High"]

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",   # 学習監視用。最終判定は balanced_accuracy
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "subsample": 0.9,
        "bootstrap_type": "Bernoulli",
        "random_seed": SEED,
        "verbose": 200,
        "allow_writing_files": False,
        "task_type": "CPU",
    }


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.SEED)
print(CFG.SAVE_DIR)

/kaggle/working/exp_20260406_013_gbdt_catnum_top4_min


## 2. Load data

In [2]:
INPUT_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
# 使っていないので削除
# original = pd.read_csv("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")

print(train.shape, test.shape)
display(train.head())

(630000, 21) (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## 3. Basic checks

In [3]:
print("train target unique:", train[CFG.TARGET_COL].value_counts(dropna=False).to_dict())
print("top4 high-IV cols:", CFG.HIGH_IV_TOP4)

train target unique: {'Low': 369917, 'Medium': 239074, 'High': 21009}
top4 high-IV cols: ['Crop_Growth_Stage', 'Water_Source', 'Crop_Type', 'Irrigation_Type']


## 4. CATNUM feature builder

In [4]:
def build_catnum_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
    stats: list[str],
    target_col: str,
    id_col: str,
    fillna_value: float = -9999.0,
):
    """
    train + test をまとめて group stats を作るのは leakage ではない。
    target を使わない純粋な test-time available feature だから。
    ただし元 train/test 分布差は乗るので、今回は最小検証として採用。
    """
    train_x = train_df.drop(columns=[target_col]).copy()
    test_x = test_df.copy()

    whole = pd.concat(
        [
            train_x.assign(__source="train"),
            test_x.assign(__source="test"),
        ],
        axis=0,
        ignore_index=True,
    )

    created_cols = []

    for cat in cat_cols:
        grp = whole.groupby(cat, dropna=False)

        agg_df = grp[num_cols].agg(stats)
        agg_df.columns = [f"CATNUM__{cat}__{num}__{stat}" for num, stat in agg_df.columns]
        agg_df = agg_df.reset_index()

        whole = whole.merge(agg_df, on=cat, how="left")
        created_cols.extend([c for c in agg_df.columns if c != cat])

    whole[created_cols] = whole[created_cols].fillna(fillna_value)

    train_feat = whole.loc[whole["__source"] == "train"].drop(columns=["__source"]).reset_index(drop=True)
    test_feat = whole.loc[whole["__source"] == "test"].drop(columns=["__source"]).reset_index(drop=True)

    return train_feat, test_feat, created_cols

## 5. Build features

In [5]:
train_feat, test_feat, catnum_cols = build_catnum_features(
    train_df=train,
    test_df=test,
    cat_cols=CFG.HIGH_IV_TOP4,
    num_cols=CFG.NUM_COLS,
    stats=CFG.CATNUM_STATS,
    target_col=CFG.TARGET_COL,
    id_col=CFG.ID_COL,
    fillna_value=CFG.FILLNA_VALUE,
)

print("n_catnum_cols =", len(catnum_cols))
print(catnum_cols[:10])

display(train_feat.head())

n_catnum_cols = 220
['CATNUM__Crop_Growth_Stage__Soil_pH__mean', 'CATNUM__Crop_Growth_Stage__Soil_pH__std', 'CATNUM__Crop_Growth_Stage__Soil_pH__median', 'CATNUM__Crop_Growth_Stage__Soil_pH__min', 'CATNUM__Crop_Growth_Stage__Soil_pH__max', 'CATNUM__Crop_Growth_Stage__Soil_Moisture__mean', 'CATNUM__Crop_Growth_Stage__Soil_Moisture__std', 'CATNUM__Crop_Growth_Stage__Soil_Moisture__median', 'CATNUM__Crop_Growth_Stage__Soil_Moisture__min', 'CATNUM__Crop_Growth_Stage__Soil_Moisture__max']


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,CATNUM__Irrigation_Type__Field_Area_hectare__mean,CATNUM__Irrigation_Type__Field_Area_hectare__std,CATNUM__Irrigation_Type__Field_Area_hectare__median,CATNUM__Irrigation_Type__Field_Area_hectare__min,CATNUM__Irrigation_Type__Field_Area_hectare__max,CATNUM__Irrigation_Type__Previous_Irrigation_mm__mean,CATNUM__Irrigation_Type__Previous_Irrigation_mm__std,CATNUM__Irrigation_Type__Previous_Irrigation_mm__median,CATNUM__Irrigation_Type__Previous_Irrigation_mm__min,CATNUM__Irrigation_Type__Previous_Irrigation_mm__max
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,7.452692,4.209838,7.32,0.3,15.0,62.114570,33.965505,60.78,0.02,119.99
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,7.532536,4.228257,7.41,0.3,15.0,61.847992,34.263488,60.03,0.02,119.99
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,7.453320,4.227406,7.30,0.3,15.0,61.995859,33.799461,61.26,0.02,119.99
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,7.617215,4.205402,7.49,0.3,15.0,63.324016,34.884325,61.89,0.02,119.99
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,7.617215,4.205402,7.49,0.3,15.0,63.324016,34.884325,61.89,0.02,119.99


## 6. Feature list

In [6]:
feature_cols = [c for c in train_feat.columns if c not in [CFG.ID_COL, CFG.TARGET_COL]]
cat_features = [c for c in CFG.BASE_CAT_COLS if c in feature_cols]

print("n_features =", len(feature_cols))
print("n_cat_features =", len(cat_features))
print("n_num_like_features =", len(feature_cols) - len(cat_features))
print("cat_features =", cat_features)

n_features = 239
n_cat_features = 8
n_num_like_features = 231
cat_features = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Region', 'Mulching_Used']


## 7. CV training

In [7]:
def fit_catboost_cv(train_df, test_df, y, feature_cols, cat_features):
    n_classes = y.nunique()
    oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
    test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)

    fold_scores = []
    class_names = None

    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    X = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_va = y.iloc[va_idx].copy()

        model = CatBoostClassifier(**CFG.CATBOOST_PARAMS)

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            use_best_model=True,
        )

        if class_names is None:
            class_names = list(model.classes_)

        va_pred = model.predict(X_va).reshape(-1)
        va_proba = model.predict_proba(X_va)
        te_proba = model.predict_proba(X_test)

        score = balanced_accuracy_score(y_va, va_pred)
        fold_scores.append(score)

        oof_proba[va_idx] = va_proba
        test_proba += te_proba / CFG.N_SPLITS

        print(f"fold {fold} balanced_accuracy = {score:.6f}")

        del model, X_tr, y_tr, X_va, y_va, va_pred, va_proba, te_proba
        gc.collect()

    cv_score = float(np.mean(fold_scores))
    return oof_proba, test_proba, fold_scores, cv_score, class_names

## 8. Run CV

In [8]:
oof_proba, test_proba, fold_scores, cv_score, class_names = fit_catboost_cv(
    train_df=train_feat,
    test_df=test_feat,
    y=train[CFG.TARGET_COL].copy(),
    feature_cols=feature_cols,
    cat_features=cat_features,
)

print("class_names =", class_names)
print("fold_scores =", fold_scores)
print("cv_score =", cv_score)

0:	learn: 0.9801896	test: 0.9804778	best: 0.9804778 (0)	total: 1.75s	remaining: 1h 27m 34s
200:	learn: 0.9839391	test: 0.9841397	best: 0.9841799 (193)	total: 5m 17s	remaining: 1h 13m 44s
400:	learn: 0.9841153	test: 0.9842522	best: 0.9842769 (267)	total: 10m	remaining: 1h 4m 49s
600:	learn: 0.9842274	test: 0.9843160	best: 0.9843160 (588)	total: 14m 15s	remaining: 56m 54s
800:	learn: 0.9843239	test: 0.9843400	best: 0.9843718 (730)	total: 18m 44s	remaining: 51m 26s
1000:	learn: 0.9844147	test: 0.9843487	best: 0.9843884 (877)	total: 23m 15s	remaining: 46m 26s
1200:	learn: 0.9844746	test: 0.9843888	best: 0.9843971 (1195)	total: 27m 39s	remaining: 41m 26s
1400:	learn: 0.9845730	test: 0.9844209	best: 0.9844374 (1384)	total: 32m 9s	remaining: 36m 42s
1600:	learn: 0.9846289	test: 0.9844208	best: 0.9844612 (1508)	total: 36m 34s	remaining: 31m 57s
1800:	learn: 0.9847011	test: 0.9844531	best: 0.9844612 (1508)	total: 41m 7s	remaining: 27m 22s
2000:	learn: 0.9847675	test: 0.9844607	best: 0.9844612 (

## 9. Save artifacts

In [9]:
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_proba)

oof_pred_idx = oof_proba.argmax(axis=1)
oof_pred_label = pd.Series([class_names[i] for i in oof_pred_idx])

oof_df = pd.DataFrame({
    CFG.ID_COL: train[CFG.ID_COL],
    "y_true": train[CFG.TARGET_COL],
    "y_pred": oof_pred_label,
})
oof_df.to_csv(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_labels.csv", index=False)

sub = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET_COL: [class_names[i] for i in test_proba.argmax(axis=1)]
})
sub.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

feature_columns_df = pd.DataFrame({
    "feature": feature_cols,
    "is_base_cat": [c in cat_features for c in feature_cols],
    "is_catnum": [c in catnum_cols for c in feature_cols],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "cv_score": cv_score,
            "fold_scores": fold_scores,
            "class_names": class_names,
            "n_features": len(feature_cols),
            "n_cat_features": len(cat_features),
            "n_catnum_cols": len(catnum_cols),
            "high_iv_top4": CFG.HIGH_IV_TOP4,
            "catnum_stats": CFG.CATNUM_STATS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

## 10. Quick summary

In [10]:
summary_df = pd.DataFrame({
    "item": [
        "cv_score",
        "n_features",
        "n_cat_features",
        "n_catnum_cols",
    ],
    "value": [
        cv_score,
        len(feature_cols),
        len(cat_features),
        len(catnum_cols),
    ]
})
display(summary_df)

,item,value
0,cv_score,0.961637
1,n_features,239.000000
2,n_cat_features,8.000000
3,n_catnum_cols,220.000000
